### Load and split the data

In [ ]:
from neurovision.preprocessing import prepare_data, make_dataset

data = prepare_data("../data/raw")
X_train, y_train, p_train = data["train"]
X_val, y_val, p_val = data["val"]
X_test, y_test, p_test = data["test"]

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

### Build the CNN and inspect it

In [ ]:
from neurovision.model import build_model

model = build_model()
model.summary()

### Class weights computed on the training set only

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from neurovision.preprocessing import encoder

y_train_idx = encoder.transform(y_train)
weights = compute_class_weight("balanced", classes=np.arange(4), y=y_train_idx)
class_weight = dict(enumerate(weights))
print(class_weight)

### Callbacks: save the best model, stop early, resume after interruption

In [ ]:
import tensorflow as tf

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "../models/best_model.keras", monitor="val_accuracy", save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.BackupAndRestore(backup_dir="../models/backup"),
]

### Train the model and measure the training time

In [ ]:
import time

EPOCHS = 1

start = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)
duration = time.time() - start
print(f"Training time: {duration / 60:.1f} min for {len(history.history['loss'])} epoch(s)")

### Learning curves: accuracy and loss

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.savefig("../reports/learning_curves.png", dpi=150)
plt.show()